In [1]:
#图11
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# --- 定义饼图绘制函数 ---
# 新增 show_overall_legend_components 参数，默认为 True
def plot_pie_chart(process_counts, title, output_path, show_legend=True, show_overall_legend_components=True):
    """
    绘制表示群落构建过程的嵌套饼图。

    参数:
    process_counts (dict): 一个字典，键是生态过程的短名称 (如 'HeS', 'DR')，值是对应的计数。
    title (str): 饼图的标题。
    output_path (str): 保存饼图的完整文件路径。
    show_legend (bool): 是否显示整个图例框。True 显示, False 不显示。
    show_overall_legend_components (bool): 仅当 show_legend 为 True 时有效。
                                           是否在图例中包含 'Deterministic Process' 和 'Stochastic Process'。
                                           True 包含, False 不包含。
    """
    all_process_types = {
        'HeS': 0, 'HoS': 0, 'DL': 0, 'HD': 0, 'DR': 0
    }
    all_process_types.update(process_counts)

    deterministic_hetero_size = all_process_types['HeS']
    deterministic_homo_size = all_process_types['HoS']
    dispersal_limitation_size = all_process_types['DL']
    homogenizing_dispersal_size = all_process_types['HD']
    drift_size = all_process_types['DR']

    total_elements = sum(all_process_types.values())
    if total_elements == 0:
        print(f"Warning: No elements found for {title}. Skipping pie chart generation.")
        return

    fig, ax = plt.subplots(figsize=(10, 10)) # 进一步增大图片整体尺寸
    ax.axis('equal')

    # 定义统一的百分比字体大小
    percentage_fontsize = 20

    outer_sizes = [deterministic_hetero_size + deterministic_homo_size,
                   dispersal_limitation_size + homogenizing_dispersal_size + drift_size]
    outer_colors = ['#CD6155', '#5499C7']
    outer_wedgeprops = {'width': 0.3, 'edgecolor': 'white'}

    valid_outer_sizes = [s for s in outer_sizes if s > 0]
    valid_outer_colors = [color for i, color in enumerate(outer_colors) if outer_sizes[i] > 0]

    # 绘制外层饼图。不传递 labels 参数。
    ax.pie(valid_outer_sizes, colors=valid_outer_colors,
           wedgeprops=outer_wedgeprops, startangle=90, counterclock=False,
           autopct=lambda p: '{:.1f}%'.format(p) if p > 0 else '',
           pctdistance=0.85,
           textprops={'fontsize': percentage_fontsize, 'color': 'black'}) # 使用统一的百分比字体大小

    inner_sizes = [deterministic_hetero_size, deterministic_homo_size,
                   dispersal_limitation_size, homogenizing_dispersal_size, drift_size]
    inner_labels = ['Heterogeneous Selection', 'Homogeneous Selection',
                    'Dispersal Limitation', 'Homogenizing Dispersal', 'Drift']
    inner_colors = ['#E59866', '#F1C40F', '#C39BD3', '#85C1E9', '#73C6B6']
    inner_wedgeprops = {'width': 0.4, 'edgecolor': 'white'}

    valid_inner_sizes = [s for s in inner_sizes if s > 0]
    valid_inner_colors = [color for i, color in enumerate(inner_colors) if inner_sizes[i] > 0]

    wedges, texts, autotexts = ax.pie(valid_inner_sizes, colors=valid_inner_colors, radius=0.7,
                                     startangle=90, counterclock=False,
                                     autopct=lambda p: '{:.1f}%'.format(p) if p > 0 else '',
                                     pctdistance=0.75,
                                     textprops={'fontsize': percentage_fontsize, 'color': 'black'}, # 使用统一的百分比字体大小
                                     wedgeprops=inner_wedgeprops)

    for i, autotext in enumerate(autotexts):
        if i < len(valid_inner_sizes) and valid_inner_sizes[i] > 0:
            autotext.set_color('black')

    # --- 创建自定义图例 (根据 show_legend 参数控制是否显示) ---
    if show_legend:
        legend_elements = []
        if show_overall_legend_components:
            legend_elements.append(mpatches.Patch(color=outer_colors[0], label='Deterministic Process'))
            legend_elements.append(mpatches.Patch(color=outer_colors[1], label='Stochastic Process'))

        for i, label in enumerate(inner_labels):
            legend_elements.append(mpatches.Patch(color=inner_colors[i], label=label))

        ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.05),
                  fontsize=13, frameon=False, ncol=1) # 图例字体进一步加大，保持1列

        plt.tight_layout(rect=[0, 0.1, 1, 1])
    else:
        plt.tight_layout()

    # 设置饼图标题
    # 标题字体进一步加大，y轴位置再次调整使其更靠近图片
    ax.set_title(title, fontsize=30, y=0.95)
    
    b_output_dir_parent = os.path.dirname(output_path)
    if not os.path.exists(b_output_dir_parent):
        os.makedirs(b_output_dir_parent)
    plt.savefig(output_path, bbox_inches='tight', dpi=300)
    plt.close(fig)

#细菌生态过程饼图
# --- 主程序逻辑 ---
# IMPORTANT: 请将此路径调整为您的 R 脚本保存 'iCAMP_all_combined.csv' 的实际数据目录！
b_base_dir = "/mnt/d/study/master/iCAMP/bacteria/"
b_output_base_dir = "/mnt/d/study/master/Extended_Data_tables/Figure_11/"
b_combined_icamp_file = os.path.join(b_base_dir, "iCAMP_all_combined.csv")

# 定义组名到完整标题的映射
group_title_map = {
    "JRG": "Jurong Rhizosphere Soil",
    "JJG": "Jingjiang Rhizosphere Soil",
    "TZG": "Tongzhou Rhizosphere Soil",
    "PAG": "Panan Rhizosphere Soil",
    "JRN": "Jurong Bulb",
    "JJN": "Jingjiang Bulb",
    "TZN": "Tongzhou Bulb",
    "PAN": "Panan Bulb"
}

# 检查合并文件是否存在。
if not os.path.exists(b_combined_icamp_file):
    print(f"Error: The combined iCAMP file was not found at {b_combined_icamp_file}.")
    print("Please ensure your R script has successfully generated this file.")
else:
    print(f"Reading combined iCAMP data from: {b_combined_icamp_file}")
    try:
        b_df_combined = pd.read_csv(b_combined_icamp_file)

        if 'process' not in b_df_combined.columns or 'group' not in b_df_combined.columns:
            print(f"Error: 'process' or 'group' column not found in {b_combined_icamp_file}.")
            print("Please check the column names in your CSV file.")
        else:
            # --- 生成第一张整体图：包含所有图例 (包括 Deterministic 和 Stochastic) ---
            print("Processing: Overall Ecological processes for all taxa (with all legend components)")
            b_overall_process_counts = b_df_combined['process'].value_counts().to_dict()
            b_overall_output_file_with_legend = os.path.join(b_output_base_dir, "b_Overall_ecological_processes_all_taxa_with_legend.pdf")
            
            # 整体标题，添加换行符
            overall_title_with_linebreak = "Ecological processes for\nall bacterial taxa"
            
            plot_pie_chart(b_overall_process_counts, overall_title_with_linebreak, b_overall_output_file_with_legend,
                           show_legend=True, show_overall_legend_components=True)
            print(f"Generated Overall pie chart with all legend components at: {b_overall_output_file_with_legend}")

            # --- 生成第二张整体图：完全没有图例 ---
            print("Processing: Overall Ecological processes for all taxa (no legend)")
            b_overall_output_file_no_legend = os.path.join(b_output_base_dir, "b_Overall_ecological_processes_all_taxa_no_legend.pdf")
            
            # 整体标题，添加换行符
            overall_title_with_linebreak = "Ecological processes for\nall bacterial taxa"
            
            plot_pie_chart(b_overall_process_counts, overall_title_with_linebreak, b_overall_output_file_no_legend,
                           show_legend=False) # show_legend=False 会禁用所有图例
            print(f"Generated Overall pie chart with no legend at: {b_overall_output_file_no_legend}")

            # --- 生成 Group-Specific Pie Charts (使用新的分组标题，不显示任何图例) ---
            b_group_names = b_df_combined['group'].unique().tolist()
            print("Found individual groups in combined data:", b_group_names)

            for group_name in b_group_names:
                print(f"Processing group: {group_name}")
                b_df_group = b_df_combined[b_df_combined['group'] == group_name].copy()
                b_process_counts = b_df_group['process'].value_counts().to_dict()
                
                group_title = group_title_map.get(group_name, f'{group_name} Community Assembly')
                
                b_output_file = os.path.join(b_output_base_dir, f"b_{group_name}_community_assembly_pie.pdf")
                # 分组图完全不显示图例
                plot_pie_chart(b_process_counts, group_title, b_output_file, show_legend=False)
                print(f"Generated {group_name} pie chart at: {b_output_file}")

    except Exception as e:
        print(f"An error occurred while processing {b_combined_icamp_file}: {e}")
        import traceback
        traceback.print_exc()

print("All pie charts generation process finished.")



#真菌生态过程饼图
# --- 主程序逻辑 ---
# IMPORTANT: 请将此路径调整为您的 R 脚本保存 'iCAMP_all_combined.csv' 的实际数据目录！
f_base_dir = "/mnt/d/study/master/iCAMP/fungi/"
f_output_base_dir = "/mnt/d/study/master/Extended_Data_tables/Figure_11/"
f_combined_icamp_file = os.path.join(f_base_dir, "iCAMP_all_combined.csv")

# 定义组名到完整标题的映射
group_title_map = {
    "JRG": "Jurong Rhizosphere Soil",
    "JJG": "Jingjiang Rhizosphere Soil",
    "TZG": "Tongzhou Rhizosphere Soil",
    "PAG": "Panan Rhizosphere Soil",
    "JRN": "Jurong Bulb",
    "JJN": "Jingjiang Bulb",
    "TZN": "Tongzhou Bulb",
    "PAN": "Panan Bulb"
}

# 检查合并文件是否存在。
if not os.path.exists(f_combined_icamp_file):
    print(f"Error: The combined iCAMP file was not found at {f_combined_icamp_file}.")
    print("Please ensure your R script has successfully generated this file.")
else:
    print(f"Reading combined iCAMP data from: {f_combined_icamp_file}")
    try:
        f_df_combined = pd.read_csv(f_combined_icamp_file)

        if 'process' not in f_df_combined.columns or 'group' not in f_df_combined.columns:
            print(f"Error: 'process' or 'group' column not found in {f_combined_icamp_file}.")
            print("Please check the column names in your CSV file.")
        else:
            # --- 生成第一张整体图：包含所有图例 (包括 Deterministic 和 Stochastic) ---
            print("Processing: Overall Ecological processes for all taxa (with all legend components)")
            f_overall_process_counts = f_df_combined['process'].value_counts().to_dict()
            f_overall_output_file_with_legend = os.path.join(f_output_base_dir, "f_Overall_ecological_processes_all_taxa_with_legend.pdf")
            
            # 整体标题，添加换行符
            overall_title_with_linebreak = "Ecological processes for\nall fungal taxa"
            
            plot_pie_chart(f_overall_process_counts, overall_title_with_linebreak, f_overall_output_file_with_legend,
                           show_legend=True, show_overall_legend_components=True)
            print(f"Generated Overall pie chart with all legend components at: {f_overall_output_file_with_legend}")

            # --- 生成第二张整体图：完全没有图例 ---
            print("Processing: Overall Ecological processes for all taxa (no legend)")
            f_overall_output_file_no_legend = os.path.join(f_output_base_dir, "f_Overall_ecological_processes_all_taxa_no_legend.pdf")
            
            # 整体标题，添加换行符
            overall_title_with_linebreak = "Ecological processes for\nall fungal taxa"
            
            plot_pie_chart(f_overall_process_counts, overall_title_with_linebreak, f_overall_output_file_no_legend,
                           show_legend=False) # show_legend=False 会禁用所有图例
            print(f"Generated Overall pie chart with no legend at: {f_overall_output_file_no_legend}")

            # --- 生成 Group-Specific Pie Charts (使用新的分组标题，不显示任何图例) ---
            f_group_names = f_df_combined['group'].unique().tolist()
            print("Found individual groups in combined data:", f_group_names)

            for group_name in f_group_names:
                print(f"Processing group: {group_name}")
                f_df_group = f_df_combined[f_df_combined['group'] == group_name].copy()
                f_process_counts = f_df_group['process'].value_counts().to_dict()
                
                group_title = group_title_map.get(group_name, f'{group_name} Community Assembly')
                
                f_output_file = os.path.join(f_output_base_dir, f"f_{group_name}_community_assembly_pie.pdf")
                # 分组图完全不显示图例
                plot_pie_chart(f_process_counts, group_title, f_output_file, show_legend=False)
                print(f"Generated {group_name} pie chart at: {f_output_file}")

    except Exception as e:
        print(f"An error occurred while processing {f_combined_icamp_file}: {e}")
        import traceback
        traceback.print_exc()

print("All pie charts generation process finished.")


Reading combined iCAMP data from: /mnt/d/study/master/iCAMP/bacteria/iCAMP_all_combined.csv
Processing: Overall Ecological processes for all taxa (with all legend components)
Generated Overall pie chart with all legend components at: /mnt/d/study/master/Extended_Data_tables/Figure_11/b_Overall_ecological_processes_all_taxa_with_legend.pdf
Processing: Overall Ecological processes for all taxa (no legend)
Generated Overall pie chart with no legend at: /mnt/d/study/master/Extended_Data_tables/Figure_11/b_Overall_ecological_processes_all_taxa_no_legend.pdf
Found individual groups in combined data: ['JJG', 'JJN', 'JRG', 'JRN', 'PAG', 'PAN', 'TZG', 'TZN']
Processing group: JJG
Generated JJG pie chart at: /mnt/d/study/master/Extended_Data_tables/Figure_11/b_JJG_community_assembly_pie.pdf
Processing group: JJN
Generated JJN pie chart at: /mnt/d/study/master/Extended_Data_tables/Figure_11/b_JJN_community_assembly_pie.pdf
Processing group: JRG
Generated JRG pie chart at: /mnt/d/study/master/Exte